# 모스 (Mo's)

`-` 정적 배열에서 여러 개의 쿼리를 제곱근 분할법을 기반으로 정렬한 뒤 처리하는 알고리즘

## 수열과 쿼리 5

- 문제 출처: [백준 13547번](https://www.acmicpc.net/problem/13547)

`-` 일반적인 세그먼트 트리를 사용해서 풀기에는 난감하다

`-` 임의의 구간에 존재하는 서로 다른 수의 개수를 세야 하기에 노드별로 구간에 어떤 수가 있는지 저장해야 한다

`-` 하지만 이러면 취합하는 데 최악의 경우 $O(N)$의 시간 복잡도를 가지게 된다

`-` 어떤 방법이 있는지 공부하고 오자

`-` Mo's 알고리즘을 통해 해결할 수 있는 문제라고 한다

`-` 원소가 $N$개 있는데 이를 $P$개의 그룹으로 쪼갠다고 해보자

`-` 그럼 각 그룹엔 $\frac{N}{P}$개의 원소가 있다

`-` 그리고 각 그룹마다 어떤 수가 몇 개 있는지와 서로 다른 수의 개수를 저장하자

`-` 쿼리가 주어졌는데 범위가 첫 번째 그룹만을 가리킨다고 해보자

`-` 원래라면 $O\left(\frac{N}{P}\right)$의 시간 복잡도를 가지겠지만 여기선 $O(1)$에 처리할 수 있다

`-` 하지만 쿼리 범위가 항상 깔끔하게 주어지는 건 아니다

`-` 시간 복잡도는 그룹 내 원소의 개수와 그룹의 개수에 영향을 받는다

`-` 즉, $O\left(\max\left(\frac{N}{P}, P\right)\right)$이다

`-` 따라서 $\frac{N}{P}= P$를 만족해야 시간 복잡도가 최소이고 이때의 $P=\sqrt{N}$이다

`-` 이것이 제곱근 분할법이다

`-` 근데 이것만으론 문제를 해결할 수 없다

`-` 쿼리 범위가 좁으면 괜찮지만 넓다면 서로 다른 수의 개수를 세는 데 $O(N)$의 시간 복잡도를 가진다

`-` 잘 생각해보면 굳이 쿼리를 입력받은 순서대로 처리할 필요가 없다

`-` 출력만 입력받은 순서대로 하면 되는 것이다!

`-` 모든 $M$개의 쿼리에 대해 $j$를 포함하는 그룹의 인덱스를 기준으로 오름차순 정렬하자

`-` 만약 동일하다면 $i$를 기준으로 오름차순 정렬하자

`-` 이제 첫 번째 쿼리부터 처리해보자

`-` 처음엔 순회하며 원소를 세면 된다

`-` 이는 $O(N)$이다

`-` 하지만 그 다음 쿼리부턴 빠르게 처리할 수 있다

`-` $j$가 같은 그룹에 속하므로 이를 조정하는 건 $O\left(\sqrt{N}\right)$이다

`-` 이제 $i$를 조정해보자

`-` $j$가 같은 그룹에 속하는 모든 쿼리를 생각해보자

`-` $i$를 기준으로 오름차순 정렬했으므로 조정할 원소의 개수는 최대 $O(N)$이다

`-` 이는 $j$가 포함되는 그룹이 바뀔 때만 발생하므로 총 $O\left(N\sqrt{N}\right)$이다

`-` 만약 $j$가 속한 그룹이 바뀌어야 한다면 초기화하면 된다

`-` 초기화하면 다음 쿼리부턴 순회하며 원소를 세야 하지만 이는 그룹이 바뀔 때 $1$번만 발생하며 그룹의 개수는 $O\left(\sqrt{N}\right)$이다

`-` 따라서 전체 알고리즘의 시간 복잡도는 $O\left((N+M)\sqrt{N} + M\log M\right)$이다

`-` $\log M \le \sqrt{N}$일테니 $O\left((N+M)\sqrt{N}\right)$로 표현해도 괜찮다

`-` 참고: https://infossm.github.io/blog/2019/02/09/mo's-algorithm/

`-` 참고: https://infossm.github.io/blog/2021/07/19/distinct-value-query/

`-` 사실 그룹별로 최솟값, 구간 합 같은 통계량을 가지고 있을 필요는 없다

`-` 그래서 단순히 정렬된 쿼리마다 이전과 차이나는 만큼 원소를 추가하거나 제거하고 이를 쿼리 결과에 반영하면 된다

`-` 즉, 단순히 쿼리 순서만 바꿨을 뿐인데 시간 복잡도가 $O(NM)$에서  $O\left((N+M)\sqrt{N}\right)$으로 줄어든 것이다!

In [55]:
import math


def compress(array):
    unique = set(array)
    mapping = {a: i for i, a in enumerate(unique)}
    return [mapping[a] for a in array]


def mos(array, quries):
    global RESULT
    n = len(array)
    m = len(quries)
    array = compress(array)
    block_size = math.ceil(n**0.5)
    quries.sort(key=lambda q: (q[1] // block_size, q[0]))
    counts = [0] * n
    answers = [0] * m
    RESULT = 0
    i_prev, j_prev = 0, -1
    for i, j, k in quries:
        while i < i_prev:
            i_prev -= 1
            add(array[i_prev], counts)
        while j > j_prev:
            j_prev += 1
            add(array[j_prev], counts)
        while i > i_prev:
            remove(array[i_prev], counts)
            i_prev += 1
        while j < j_prev:
            remove(array[j_prev], counts)
            j_prev -= 1
        i_prev, j_prev = i, j
        answers[k] = RESULT
    return answers


def add(a, counts):
    global RESULT
    if counts[a] == 0:
        RESULT += 1
    counts[a] += 1


def remove(a, counts):
    global RESULT
    counts[a] -= 1
    if counts[a] == 0:
        RESULT -= 1


def solution():
    N = int(input())
    array = list(map(int, input().split()))
    M = int(input())
    quries = []
    for k in range(M):
        i, j = map(int, input().split())
        quries.append((i - 1, j - 1, k))
    answers = mos(array, quries)
    print("\n".join(map(str, answers)))


solution()

# input
# 5
# 1 1 2 1 3
# 3
# 1 5
# 2 4
# 3 5

 5
 1 1 2 1 3
 3
 1 5
 2 4
 3 5


3
2
3


`-` [서로 다른 수와 쿼리 1](https://www.acmicpc.net/problem/14897) 날먹 ㄱㄱ

`-` [민호의 소원](https://www.acmicpc.net/problem/13028)이랑 [Poklon](https://www.acmicpc.net/problem/14413)도 같이 ㄱㄱ